# Imports

In [1]:
from science_jubilee.Machine import Machine
from xarm.wrapper import XArmAPI
import numpy as np

SDK_VERSION: 1.18.5


# Connecting to Jubilee

In [ ]:
m = Machine(address='192.168.1.2')

# Connecting to XArm

In [ ]:
arm = XArmAPI('192.168.1.205') #Arm is named arm

# Arm Setup and Enabling

In [ ]:
arm.motion_enable(enable=True) #enabling motion on arm
arm.set_world_offset([0,0,8,0,0,0]) #8 mm offset due to mounting plate
arm.set_tcp_offset([119.569, 0, 12.6, 0, 0, -1]) #tool coordinates offset to tip of tool
arm.set_state(state=0)
arm.set_tcp_load(weight = 0.4997 + 0.048, center_of_gravity=[41,0,18]) #weight of tool in kg. Center of gravity in mm
arm.set_mode(0)  #Mode 0, positional control
arm.set_state(state=0)

# Functions for Bed Movement

In [ ]:
def zprobe(count):
    """
    Detect height of object by probing in the z direction.

    count: number of probes
    """
    readings = np.zeros(count)
    for i in range(count):
        while True:
            if arm.get_tgpio_digital(ionum = 1)[1] == 1:
                readings[i] = arm.get_position()[1][2]
                arm.set_tool_position(z = -5, speed = 50, wait = True)
                break
            else:
                arm.set_tool_position(z = 0.5, speed = 5, wait = True)
    return np.mean(readings), np.std(readings), readings


def zprobeRel(count):
    """
    Detect height of object relative to tool by probing in the z direction.

    count: number of probes
    """
    readings = np.zeros(count)

    for i in range(count):
        distance = 0
        while True:
            if arm.get_tgpio_digital(ionum = 1)[1] == 1:
                readings[i] = distance
                arm.set_tool_position(z = -1 * distance, speed = 50, wait = True)
                break
            else:
                arm.set_tool_position(z = 0.5, speed = 5, wait = True)
                distance += 0.5
    return np.mean(readings), np.std(readings), readings

def xprobe(count):
    """
    Detect position of object by probing in the tool's x direction; move the arm closer to the object in x dir

    count: number of probes
    """
    readings = np.zeros(count)
    for i in range(count):
        while True:
            if arm.get_tgpio_digital(ionum = 0)[1] == 1:
                readings[i] = arm.get_position()[1][1]
                arm.set_tool_position(x = -5, speed = 50, wait = True)
                break
            else:
                arm.set_tool_position(x = 0.5, speed = 5, wait = True)
    return np.mean(readings), np.std(readings), readings

def xprobeRel(count):
    """
    Detect position of object relative to tool by probing in the tool's x direction.

    count: number of probes
    """
    readings = np.zeros(count)
    for i in range(count):
        distance = 0
        while True:
            if arm.get_tgpio_digital(ionum = 0)[1] == 1:
                readings[i] = distance
                arm.set_tool_position(x = -1 * distance, speed = 50, wait = True)
                break
            else:
                arm.set_tool_position(x = 0.5, speed = 5, wait = True)
                distance += 0.5
    return np.mean(readings), np.std(readings), readings

def findYaw():
    """
    Adjust yaw of end effector based on position of plate.
    """
    xprobe(1)
    arm.set_tool_position(y = 5, speed = 50, wait = True)
    p1 = xprobeRel(1)[0]
    arm.set_tool_position(y = -10, speed = 50, wait = True)
    p2 = xprobeRel(1)[0]
    arm.set_tool_position(x = -5, y = 5, speed = 50, wait = True)
    arm.set_tool_position(yaw = -180*np.tan((p1 - p2)/10)/np.pi, wait = True)
    arm.set_tool_position(y = 80, speed = 100, wait = True)
    p3 = xprobeRel(1)[0]
    arm.set_tool_position(y = -160, speed = 100, wait = True)
    p4 = xprobeRel(1)[0]
    arm.set_tool_position(x = -10, y = 80, speed = 100, wait = True)
    arm.set_tool_position(yaw = -180*np.tan((p3 - p4)/160)/np.pi, wait = True)
    return -180*np.tan((p1 - p2)/10)/np.pi, -180*np.tan((p3 - p4)/160)/np.pi

def findpoint():
    """
    Detect raised point on buildplate.
    """
    xprobe(1)
    for i in [2, 1, 0.5]:
        x1 = xprobeRel(1)[0]
        y1 = 0
        while True:
            arm.set_tool_position(y = i, wait = True)
            y1 += i
            x2 = xprobeRel(1)[0]
            if x1-x2 >= 1.5:
                print("Detected")
                arm.set_tool_position(y = -i*2, wait = True)
                break
            elif x1-x2 <= -1.5:
                arm.set_tool_position(x = -10, wait = True)
                arm.set_tool_position(y = -5, wait = True)
                arm.set_tool_position(x = 10, wait = True)
            
            if y1 > 50:
                print("error")
            else:
                pass
            x1 = x2
    arm.set_tool_position(y = -40, wait = True)

def zlevel():
    """
    Probe build plate in the z direction to determine roll and pitch of end effector.
    """
    xd = 60
    yd = 40
    p1 = np.array([0, 0, 0])
    p2 = np.array([xd, yd, 0])
    p3 = np.array([xd, -yd, 0])
    zprobe(1)
    arm.set_tool_position(z = -5, speed = 50, wait = True)
    p1[2] = zprobeRel(1)[0]
    arm.set_tool_position(x = xd, y = yd, speed = 50, wait = True)
    p2[2] = zprobeRel(1)[0]
    arm.set_tool_position(y = -2*yd, speed = 50, wait = True)
    p3[2] = zprobeRel(1)[0]
    arm.set_tool_position(x = -xd, y = yd, speed = 50, wait = True) # move back to original position

    n = np.cross(p2 - p1, p3 - p1)
    nn = n / np.linalg.norm(n)
    angles = 180 * (np.arcsin(nn)) / np.pi
    arm.set_tool_position(roll = angles[1], pitch = -angles[0], speed = 50, wait = True)

    return angles[1], angles[0], p1, p2, p3

def lineup():
    """
    Line up end effector with a build plate. Single plate version of lineupShelf()
    """
    arm.set_mode(2) # set tool to teaching mode to allow manual movement
    arm.set_state(0)
    arm.start_record_trajectory()
    input('Line Up Corner of Tool with Marker and Press Enter')
    arm.stop_record_trajectory()
    arm.set_mode(0)
    arm.set_state(0)
    arm.set_tool_position(x = -20, speed = 50, wait = True)
    arm.set_tool_position(z = -30, speed = 50, wait = True)
    arm.set_tool_position(x = 60, speed = 50, wait = True)
    zlevel()
    arm.set_tool_position(x = -60, speed = 50, wait = True)
    arm.set_tool_position(z = 17, speed = 50, wait = True)
    findYaw()
    arm.set_tool_position(y = 30)
    findpoint()
    arm.set_tool_position(z = 30, speed = 50, wait = True)
    return arm.get_position()[1]

def lineupShelf(shelfCount):
    """
    Find position of build plates on shelf

    shelfCount: number of build plates on shelf
    """
    positions = np.zeros((shelfCount, 6))
    arm.set_mode(2)
    arm.set_state(0)
    arm.start_record_trajectory()
    input('Line up corner of tool with marker on top plate and press enter')
    arm.stop_record_trajectory()
    arm.set_mode(0)
    arm.set_state(0)
    arm.set_tool_position(x = -20, speed = 50, wait = True)
    arm.set_tool_position(z = -30, speed = 50, wait = True)
    arm.set_tool_position(x = 60, speed = 50, wait = True)
    zlevel()
    arm.set_tool_position(x = -60, speed = 50, wait = True)
    arm.set_tool_position(z = 17, speed = 50, wait = True)
    findYaw()
    arm.set_tool_position(y = 30)
    findpoint()
    arm.set_tool_position(z = 30, speed = 50, wait = True)
    positions[0] = arm.get_position()[1]
    for i in range(1, len(positions)):
        input("next??")
        arm.set_tool_position(x = 30, speed = 50, wait = True)
        zlevel()
        arm.set_tool_position(x = -35, speed = 50, wait = True)
        arm.set_tool_position(z = 17, speed = 50, wait = True)
        findYaw()
        xprobe(1)
        arm.set_tool_position(z = 30, speed = 50, wait = True)
        positions[i] = arm.get_position()[1]
    return positions

def pickUp(position):
    """
    Pick up a build plate at a given location.

    position: position of build plate to be picked up
    """
    arm.set_position(*position, speed = 100, wait = True)
    #arm.set_tool_position(z = -2*np.cos(6*np.pi/180), speed = 100, wait = True)
    arm.set_tool_position(x = 40, speed = 100, wait = True)
    arm.set_tool_position(pitch = -6, speed = 100, wait = True)
    arm.set_tool_position(x = 42, z = -42*np.tan(6*np.pi/180), speed = 100, wait = True)
    arm.set_tool_position(z = -2, speed = 50, wait = True)
    arm.set_tool_position(pitch = 6, speed = 50, wait = True)
    arm.set_tcp_load(weight = 0.4997 + 0.048 + 1.609, center_of_gravity=[174,0,11])
    arm.set_tool_position(z = -20, speed = 10, wait = True)
    return arm.get_position()

def setDown(position):
    """
    Set down a build plate at a given location.

    position: position where build plate will be set down
    """
    arm.set_position(*(np.array(position) + [0, 0, (20 + 2*np.cos(6*np.pi/180)), 0, 0, 0]), speed = 100, wait = True)
    arm.set_tool_position(x = 40 + 45/np.cos(6*np.pi/180) + 2*np.tan(6*np.pi/180), speed = 100, wait = True)
    arm.set_tool_position(z = 20, speed = 10, wait = True)
    arm.set_tcp_load(weight = 0.4997 + 0.048, center_of_gravity=[41,0,18])
    arm.set_tool_position(pitch = -6, speed = 50, wait = True)
    arm.set_tool_position(z = 2, speed = 50, wait = True)
    arm.set_tool_position(x = -42, z = 42*np.tan(6*np.pi/180), speed = 100, wait = True)
    arm.set_tool_position(pitch = 6, speed = 100, wait = True)
    arm.set_tool_position(x = -40, speed = 100, wait = True)

def transferPlates(position1, position2, moveSpeed):
    """
    Transfer build plate between two positions.

    position1: pickup location of build plate
    position2: set down location of build plate
    moveSpeed: move speed of arm between locations
    """
    currentPosition = arm.get_position()[1]
    arm.set_tool_position(yaw = (currentPosition[5]-position1[5])/10, speed = moveSpeed, wait = True)
    pickUp(position1)
    arm.set_tool_position(x = -450, speed = moveSpeed, wait = True)
    arm.set_tool_position(yaw = (position1[5]-position2[5])/2, speed = moveSpeed, wait = True)
    arm.set_tool_position(yaw = (position1[5]-position2[5])/2, speed = moveSpeed, wait = True)
    arm.set_position(*(np.array(position2) + np.array([-350 * np.cos(position2[5] * np.pi / 180), -350 * np.sin(position2[5] * np.pi / 180), 20, 0, 0, 0])), speed = moveSpeed, wait = True)
    setDown(position2)

# Lining up with Shelf

In [ ]:
shelfPositions = lineupShelf(3)

# Lining up with Printer

In [ ]:
printPosition = lineupShelf(1)

# Demo

In [ ]:
m.home_all() # home printer if necessary

In [ ]:
m.move_to(z=240)
transferPlates(printPosition[0], shelfPositions[0], 100)
transferPlates(shelfPositions[2], printPosition[0], 100)
m.move_to(z=20)